In [ ]:
# fashion_qcnn_4class_cartan_pooling.py

import random
import numpy as np
import torch
import torch.nn as nn
import pennylane as qml

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


# ============================================================
# Config
# ============================================================

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_QUBITS = 8
IMG_SIZE = 16              # 16 x 16 = 256 = 2^8
N_CLASSES = 4

# Fashion-MNIST:
# 0 = T-shirt/top
# 1 = Trouser
# 7 = Sneaker
# 8 = Bag

FASHION_CLASSES = [0, 1, 7, 8]
CLASS_MAP = {0: 0, 1: 1, 7: 2, 8: 3}

TRAIN_PER_CLASS = 2000
VAL_PER_CLASS = 400
TEST_PER_CLASS = 300

BATCH_SIZE = 24
EPOCHS = 15
LR = 0.001

PERIODIC_BOUNDARY = False

# New architecture parameter counts
N_CONV_PARAMS = 11         # Cartan-inspired convolution block
N_POOL_PARAMS = 3          # controlled-transfer pooling block
USE_FINAL_CLASSIFIER = True


# ============================================================
# Reproducibility
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# Dataset utilities
# ============================================================

class RemapFashionMNIST(torch.utils.data.Dataset):
    def __init__(self, base_dataset, class_map):
        self.base_dataset = base_dataset
        self.class_map = class_map

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        x, y = self.base_dataset[idx]
        return x, self.class_map[int(y)]


def make_balanced_indices(dataset, class_map, n_per_class, offset_per_class=0):
    buckets = {c: [] for c in class_map.keys()}

    for idx in range(len(dataset)):
        _, y = dataset[idx]
        y = int(y)

        if y in buckets:
            buckets[y].append(idx)

    selected = []
    rng = np.random.default_rng(SEED)

    for c in class_map.keys():
        indices = np.array(buckets[c])
        rng.shuffle(indices)

        start = offset_per_class
        end = offset_per_class + n_per_class

        if end > len(indices):
            raise ValueError(
                f"Not enough samples for class {c}. "
                f"Requested indices up to {end}, available {len(indices)}."
            )

        selected.extend(indices[start:end].tolist())

    rng.shuffle(selected)
    return selected


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


def load_data():
    train_full = datasets.FashionMNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform,
    )

    test_full = datasets.FashionMNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform,
    )

    train_idx = make_balanced_indices(
        train_full,
        CLASS_MAP,
        TRAIN_PER_CLASS,
        offset_per_class=0,
    )

    val_idx = make_balanced_indices(
        train_full,
        CLASS_MAP,
        VAL_PER_CLASS,
        offset_per_class=TRAIN_PER_CLASS,
    )

    test_idx = make_balanced_indices(
        test_full,
        CLASS_MAP,
        TEST_PER_CLASS,
        offset_per_class=0,
    )

    train_ds = RemapFashionMNIST(Subset(train_full, train_idx), CLASS_MAP)
    val_ds = RemapFashionMNIST(Subset(train_full, val_idx), CLASS_MAP)
    test_ds = RemapFashionMNIST(Subset(test_full, test_idx), CLASS_MAP)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    return train_loader, val_loader, test_loader


# ============================================================
# Quantum circuit blocks: Cartan-inspired convolution + transfer pooling
# ============================================================

def cartan_convolution_block(theta, wires):
    """
    Axis-Balanced Cartan convolution block.

    The block keeps the QCNN logic of local two-qubit filters, but replaces
    Hur circuit 8 with a Cartan-inspired structure:

        local rotations -> XX/YY/ZZ entangling core -> local rotations

    Parameters:
        theta[0:4]   initial local rotations
        theta[4:7]   non-local Cartan-like interactions XX, YY, ZZ
        theta[7:11]  final local rotations

    Total: 11 parameters.
    """
    a, b = wires

    # Local pre-rotations: allow each qubit to choose a useful local basis.
    qml.RZ(theta[0], wires=a)
    qml.RY(theta[1], wires=a)
    qml.RZ(theta[2], wires=b)
    qml.RY(theta[3], wires=b)

    # Non-local Cartan-like core. These are the three principal two-qubit
    # interaction channels of a generic two-qubit unitary.
    qml.IsingXX(theta[4], wires=[a, b])
    qml.IsingYY(theta[5], wires=[a, b])
    qml.IsingZZ(theta[6], wires=[a, b])

    # Local post-rotations: make the block more flexible without using full SU(4).
    qml.RY(theta[7], wires=a)
    qml.RZ(theta[8], wires=a)
    qml.RY(theta[9], wires=b)
    qml.RZ(theta[10], wires=b)


def convolution_layer_on_wires(theta_conv, active_wires):
    """
    QCNN-style convolution layer on the currently active wires.

    The same Cartan-inspired block is shared over all pairs in the same
    convolutional layer, preserving the translational-invariance idea of QCNNs.

    For active_wires = [0,1,2,3,4,5,6,7]:
      even pairs    = (0,1), (2,3), (4,5), (6,7)
      shifted pairs = (1,2), (3,4), (5,6)

    For active_wires = [0,2,4,6]:
      even pairs    = (0,2), (4,6)
      shifted pairs = (2,4)
    """
    even_pairs = []
    shifted_pairs = []

    for i in range(0, len(active_wires) - 1, 2):
        even_pairs.append((active_wires[i], active_wires[i + 1]))

    for i in range(1, len(active_wires) - 1, 2):
        shifted_pairs.append((active_wires[i], active_wires[i + 1]))

    if PERIODIC_BOUNDARY and len(active_wires) > 2:
        shifted_pairs.append((active_wires[-1], active_wires[0]))

    for pair in even_pairs:
        cartan_convolution_block(theta_conv, pair)

    for pair in shifted_pairs:
        cartan_convolution_block(theta_conv, pair)


def transfer_pooling_pair(theta_pool, discard, keep):
    """
    Controlled-transfer pooling block.

    Before the discard qubit is ignored, part of its information is transferred
    to the retained qubit. This is still a QCNN pooling operation: the number of
    active qubits is reduced, but the retained qubit is adaptively updated using
    information from its local neighbor.

    Pair convention:
        discard = qubit that will not be used after pooling
        keep    = qubit that remains active

    Total: 3 parameters.
    """
    qml.CNOT(wires=[discard, keep])
    qml.CRY(theta_pool[0], wires=[discard, keep])
    qml.CRZ(theta_pool[1], wires=[discard, keep])
    qml.RY(theta_pool[2], wires=keep)


def transfer_pooling_layer(theta_pool, pool_pairs):
    """
    Generic pooling layer.

    Each pair is:
        (discard_qubit, keep_qubit)

    The same pooling parameters are shared across all pairs in the layer.
    """
    for discard, keep in pool_pairs:
        transfer_pooling_pair(theta_pool, discard, keep)


# ============================================================
# QCNN model: 8 -> 4 -> 2, probability readout
# ============================================================

class FashionQCNN4Class(nn.Module):
    def __init__(self):
        super().__init__()

        self.dev = qml.device("default.qubit", wires=N_QUBITS)

        # Layer 1: convolution on 8 qubits + pooling 8 -> 4
        self.theta_conv1 = nn.Parameter(0.01 * torch.randn(N_CONV_PARAMS))
        self.theta_pool1 = nn.Parameter(0.01 * torch.randn(N_POOL_PARAMS))

        # Layer 2: convolution on 4 retained qubits + pooling 4 -> 2
        self.theta_conv2 = nn.Parameter(0.01 * torch.randn(N_CONV_PARAMS))
        self.theta_pool2 = nn.Parameter(0.01 * torch.randn(N_POOL_PARAMS))

        # Optional final two-qubit classifier on the remaining 2 qubits.
        # It improves the final class separation while keeping the output quantum.
        if USE_FINAL_CLASSIFIER:
            self.theta_final = nn.Parameter(0.01 * torch.randn(N_CONV_PARAMS))
        else:
            self.theta_final = None

        # Initial active wires
        self.wires_8 = [0, 1, 2, 3, 4, 5, 6, 7]

        # After first pooling we keep [0, 2, 4, 6]
        self.wires_4 = [0, 2, 4, 6]

        # After second pooling we keep [0, 4]
        # These two qubits encode the 4 classes through:
        # class 0 -> |00>
        # class 1 -> |01>
        # class 2 -> |10>
        # class 3 -> |11>
        self.output_wires = [0, 4]

        @qml.qnode(self.dev, interface="torch", diff_method="backprop")
        def qnode(x_flat, theta_conv1, theta_pool1, theta_conv2, theta_pool2, theta_final):
            qml.AmplitudeEmbedding(
                features=x_flat,
                wires=range(N_QUBITS),
                normalize=True,
            )

            # ----------------------------
            # Layer 1: 8 -> 4
            # ----------------------------
            convolution_layer_on_wires(theta_conv1, self.wires_8)

            transfer_pooling_layer(
                theta_pool1,
                pool_pairs=[
                    (1, 0),
                    (3, 2),
                    (5, 4),
                    (7, 6),
                ],
            )

            # ----------------------------
            # Layer 2: 4 -> 2
            # ----------------------------
            convolution_layer_on_wires(theta_conv2, self.wires_4)

            transfer_pooling_layer(
                theta_pool2,
                pool_pairs=[
                    (2, 0),
                    (6, 4),
                ],
            )

            # ----------------------------
            # Final 2-qubit classifier
            # ----------------------------
            if USE_FINAL_CLASSIFIER:
                cartan_convolution_block(theta_final, self.output_wires)

            # Probability readout on 2 final qubits.
            # Returns probabilities in computational basis:
            # [P(00), P(01), P(10), P(11)]
            return qml.probs(wires=self.output_wires)

        self.qnode = qnode

    def forward(self, x):
        """
        x: image batch, shape (B, 1, 16, 16)

        returns:
          probs: shape (B, 4)

        Class-probability mapping:
          class 0 -> P(00) = Fashion-MNIST 0, T-shirt/top
          class 1 -> P(01) = Fashion-MNIST 1, Trouser
          class 2 -> P(10) = Fashion-MNIST 7, Sneaker
          class 3 -> P(11) = Fashion-MNIST 8, Bag
        """
        x_flat = x.reshape(x.shape[0], -1)

        theta_final = self.theta_final
        if theta_final is None:
            # Dummy tensor used only to satisfy the qnode signature.
            theta_final = torch.zeros(N_CONV_PARAMS, device=x.device, dtype=x.dtype)

        probs = self.qnode(
            x_flat,
            self.theta_conv1,
            self.theta_pool1,
            self.theta_conv2,
            self.theta_pool2,
            theta_final,
        )

        return probs.float()


# ============================================================
# Probability cross-entropy loss
# ============================================================

def probability_cross_entropy_loss(probs, labels, eps=1e-8):
    """
    Cross-entropy directly on measurement probabilities.

    probs: shape (B, 4), where each row is:
           [P(00), P(01), P(10), P(11)]

    labels: shape (B,), values in {0,1,2,3}

    Loss:
        L = - mean log P(y)
    """
    probs = torch.clamp(probs, min=eps, max=1.0)

    true_probs = probs[
        torch.arange(probs.shape[0], device=probs.device),
        labels,
    ]

    loss = -torch.log(true_probs).mean()
    return loss


# ============================================================
# Metrics
# ============================================================

def accuracy(probs, labels):
    preds = torch.argmax(probs, dim=1)
    return (preds == labels).float().mean().item()


def grad_norm(model):
    total = 0.0

    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.detach().pow(2).sum().item()

    return total ** 0.5


def confusion_matrix(model, loader):
    model.eval()

    cm = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.int64)

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            probs = model(x)
            preds = torch.argmax(probs, dim=1)

            for true_label, pred_label in zip(y.cpu(), preds.cpu()):
                cm[true_label, pred_label] += 1

    return cm


# ============================================================
# Training / evaluation
# ============================================================

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_grad_norm = 0.0
    total_n = 0
    n_batches = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        if is_train:
            optimizer.zero_grad()

        probs = model(x)
        loss = probability_cross_entropy_loss(probs, y)

        if is_train:
            loss.backward()

            batch_grad_norm = grad_norm(model)
            total_grad_norm += batch_grad_norm
            n_batches += 1

            optimizer.step()

        batch_size = x.shape[0]

        total_loss += loss.item() * batch_size
        total_acc += accuracy(probs.detach(), y) * batch_size
        total_n += batch_size

    avg_loss = total_loss / total_n
    avg_acc = total_acc / total_n

    if is_train:
        avg_grad_norm = total_grad_norm / max(n_batches, 1)
    else:
        avg_grad_norm = None

    return avg_loss, avg_acc, avg_grad_norm


# ============================================================
# Main
# ============================================================

def main():
    print(f"Using device: {DEVICE}")
    print("Fashion-MNIST classes:", FASHION_CLASSES)
    print("Class mapping:", CLASS_MAP)
    print("Architecture: Cartan-inspired convolution + controlled-transfer pooling")
    print("Qubit reduction: 8 -> 4 -> 2")

    train_loader, val_loader, test_loader = load_data()

    model = FashionQCNN4Class().to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    print("\nModel parameters:")
    print("theta_conv1:", model.theta_conv1.numel())
    print("theta_pool1:", model.theta_pool1.numel())
    print("theta_conv2:", model.theta_conv2.numel())
    print("theta_pool2:", model.theta_pool2.numel())
    if model.theta_final is not None:
        print("theta_final:", model.theta_final.numel())
    print("total:", sum(p.numel() for p in model.parameters()))

    best_val_acc = 0.0
    best_state = None
    best_epoch = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc, train_grad = run_epoch(
            model,
            train_loader,
            optimizer=optimizer,
        )

        val_loss, val_acc, _ = run_epoch(
            model,
            val_loader,
            optimizer=None,
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:02d} | "
            f"train loss={train_loss:.4f}, train acc={train_acc:.4f}, "
            f"grad norm={train_grad:.6e} | "
            f"val loss={val_loss:.4f}, val acc={val_acc:.4f}"
        )

    # Reload best model according to validation accuracy
    if best_state is not None:
        model.load_state_dict(best_state)

    test_loss, test_acc, _ = run_epoch(
        model,
        test_loader,
        optimizer=None,
    )

    print("\nBest validation accuracy:", best_val_acc)
    print("Best epoch:", best_epoch)
    print(f"Test loss={test_loss:.4f}, test acc={test_acc:.4f}")

    print("\nClass order:")
    print("0 -> T-shirt/top")
    print("1 -> Trouser")
    print("2 -> Sneaker")
    print("3 -> Bag")

    print("\nOutput probability mapping:")
    print("class 0 -> P(00)")
    print("class 1 -> P(01)")
    print("class 2 -> P(10)")
    print("class 3 -> P(11)")

    print("\nValidation confusion matrix:")
    print(confusion_matrix(model, val_loader))

    print("\nTest confusion matrix:")
    print(confusion_matrix(model, test_loader))


if __name__ == "__main__":
    main()


Using device: cpu
Fashion-MNIST classes: [0, 1, 7, 8]
Class mapping: {0: 0, 1: 1, 7: 2, 8: 3}
Encoding: pairwise fragment encoding with local 2x2 mean features
Architecture: Cartan-inspired convolution + controlled-transfer pooling
Qubit reduction: 8 -> 4 -> 2
Final classifier: True

Model parameters:
theta_enc: 32
theta_conv1: 11
theta_pool1: 3
theta_conv2: 11
theta_pool2: 3
theta_final: 11
total: 71
Epoch 01 | train loss=1.3069, train acc=0.3490, grad norm=6.127640e-01 | val loss=1.1637, val acc=0.4894
Epoch 02 | train loss=1.0870, train acc=0.5474, grad norm=4.909718e-01 | val loss=1.0174, val acc=0.5994
Epoch 03 | train loss=0.9733, train acc=0.6280, grad norm=4.558214e-01 | val loss=0.9334, val acc=0.6762
Epoch 04 | train loss=0.8920, train acc=0.6675, grad norm=4.421865e-01 | val loss=0.8548, val acc=0.7037
Epoch 05 | train loss=0.8168, train acc=0.7095, grad norm=4.342612e-01 | val loss=0.7866, val acc=0.7325
Epoch 06 | train loss=0.7689, train acc=0.7231, grad norm=4.242253e-01